# Spectroscopy workflow

This notebook walks through a full spectroscopy bring-up sequence for one multiplexed readout group: locate the resonators, refine their readout frequencies, identify the qubit transitions, and finish by updating the default control amplitudes. The code assumes you will inspect the plots, edit the configuration files manually where noted, and reload before continuing.

## 1. Create an `Experiment`

Start from the mux you want to characterize and load the corresponding configuration and parameter files for your system.

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect to the setup

Connecting also synchronizes the clocks, so do this before any spectroscopy measurement.

In [ ]:
# Clocks will be synced when connected
exp.connect()

## 3. Run a coarse resonator scan

Begin with a broad readout-frequency sweep on one representative qubit so you can identify the frequency region that covers all resonators in the mux.

In [ ]:
# Sweep the readout frequency to find resonator frequency
# Be sure to set the readout amplitude to a reasonable value
exp.scan_resonator_frequencies(
    exp.qubit_labels[0],  # Qubit label with which the resonator is associated
    frequency_range=np.arange(9.75, 10.75, 0.002),  # GHz
    readout_amplitude=0.1,
    save_image=True,
)

## 4. Sweep readout frequency and power

Use the coarse scan to define a narrower window, then build a 2D resonator spectroscopy map to choose a practical readout power. Convert that chosen power into the amplitude used by the lower-level scan APIs.

In [ ]:
# Choose the frequency range to cover all resonator frequencies
readout_frequency_range = np.arange(10.05, 10.55, 0.002)

In [ ]:
# Sweep the readout frequency and power
exp.resonator_spectroscopy(
    exp.qubit_labels[0],
    frequency_range=readout_frequency_range,
    power_range=np.arange(-60, 5, 5),
)

In [ ]:
# Choose the appropriate readout power and the corresponding amplitude
readout_power = -40  # dB
readout_amplitude = 10 ** (readout_power / 20)
readout_amplitude

## 5. Re-scan the resonators and map the peaks

With the readout amplitude fixed, repeat the resonator scan, extract the fitted peaks, and assign those peak frequencies back to qubit labels in the chip-specific order for this mux.

In [ ]:
# Sweep the readout frequency with the chosen power
result = exp.scan_resonator_frequencies(
    exp.qubit_labels[0],
    frequency_range=readout_frequency_range,
    readout_amplitude=readout_amplitude,
    save_image=True,
)

# Extract the peaks from the result
peaks = result["peaks"]

In [ ]:
# Map the resonator frequencies to the qubit labels
# The order is dependent on the chip design
resonator_frequencies = {
    exp.qubit_labels[0]: peaks[1],
    exp.qubit_labels[1]: peaks[3],
    exp.qubit_labels[2]: peaks[2],
    exp.qubit_labels[3]: peaks[0],
}
resonator_frequencies

## 6. Persist and refine the resonator frequencies

Write the coarse resonator frequencies into `props.yaml`, reload, and then measure the reflection coefficient for each qubit to extract a more precise fitted resonator frequency. After updating `props.yaml` again, reload once more before moving to qubit spectroscopy.

In [ ]:
# Update `resonator_frequency` in props.yaml manually and reload
exp.reload()

In [ ]:
# Fit the reflection coefficient at the resonance frequency
fine_resonator_frequencies = {}
for qubit in exp.qubit_labels:
    result = exp.measure_reflection_coefficient(
        qubit,
        readout_amplitude=readout_amplitude,
    )
    fine_resonator_frequencies[qubit] = result["f_r"]

fine_resonator_frequencies

In [ ]:
# Update props.yaml again and reload
exp.reload()

## 7. Find rough qubit frequencies

Run a first-pass control-frequency sweep for each qubit, then define a per-qubit frequency window that covers the `ge` and `ef` transitions you want to inspect more carefully.

In [ ]:
# Sweep the control frequency to find qubit frequency
# Note that photons in the resonator will shift the qubit frequency
for qubit in exp.qubit_labels:
    exp.scan_qubit_frequencies(
        qubit,
        control_amplitude=0.1,
        readout_amplitude=0.01,
    )

In [ ]:
# Choose the control frequency to cover ge, ef transitions
control_frequency_ranges = {
    exp.qubit_labels[0]: np.arange(6.5, 7.5, 0.005),
    exp.qubit_labels[1]: np.arange(7.5, 8.5, 0.005),
    exp.qubit_labels[2]: np.arange(7.5, 8.5, 0.005),
    exp.qubit_labels[3]: np.arange(6.5, 7.7, 0.005),
}

## 8. Sweep qubit frequency and fit the resonance

Use a 2D qubit spectroscopy scan to choose the operating region, measure the resonance for each qubit, and then update `props.yaml` with the fitted `qubit_frequency` values before reloading.

In [ ]:
# Sweep control frequency and power
# Note that photons in the resonator will shift the qubit frequency
for qubit in exp.qubit_labels:
    exp.qubit_spectroscopy(
        qubit,
        frequency_range=control_frequency_ranges[qubit],
        power_range=np.arange(-60, 5, 5),
        readout_amplitude=0.01,
    )

In [ ]:
# Fit the qubit frequency at the resonance frequency
for qubit in exp.qubit_labels:
    exp.measure_qubit_resonance(
        qubit,
        frequency_range=control_frequency_ranges[qubit],
        readout_amplitude=0.03,
    )

In [ ]:
# Update `qubit_frequency` in props.yaml manually and reload
exp.reload()

## 9. Validate the waveform and update control amplitudes

After the frequency updates, check the readout waveform, obtain a Rabi fit, and convert that result into default control amplitudes. Once you edit `params.yaml`, reload so the final validation uses the new values.

In [ ]:
# Check the reflection waveform of readout pulses
exp.check_waveform()

In [ ]:
# Measure the Rabi oscillation
exp.obtain_rabi_params()

In [ ]:
# Calculate the default control amplitudes
exp.calc_control_amplitudes()

In [ ]:
# Update `control_amplitude` in params.yaml manually and reload
exp.reload()

## 10. Re-check the Rabi oscillation

Run the Rabi measurement again to confirm the updated default amplitudes behave as expected.

In [ ]:
# Check the Rabi oscillation
exp.obtain_rabi_params()